## Selection of the best flood event from 10 years to model with 50 yrs max.(Flood Hydrograph)

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1. READ DATA
# -------------------------------
file = r"C:\Hec_Ras\ProJect6_big\US_River Level - Waikato River - Hamilton Traffic Br - 22nd Apr 2016 - 22nd Apr 2026.csv"

df = pd.read_csv(file)

# create datetime
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], dayfirst=True)
df = df.sort_values('datetime')
df = df.set_index('datetime')

flow = df['flvalue(m3/sec)']

# -------------------------------
# 2. DEFINE FLOOD THRESHOLD
# -------------------------------
threshold = flow.quantile(0.90)   # top 10% flows = flood condition

df['flood_flag'] = flow > threshold

# -------------------------------
# 3. SPLIT INTO EVENTS
# -------------------------------
df['event_id'] = (df['flood_flag'] != df['flood_flag'].shift()).cumsum()

events = df[df['flood_flag']].groupby('event_id')

# -------------------------------
# 4. EXTRACT HYDROGRAPHS
# -------------------------------
hydrographs = []
event_ids = []

for eid, group in events:
    if len(group) > 5:  # remove small noise events
        hydrographs.append(group['flvalue(m3/sec)'].values)
        event_ids.append(eid)

# -------------------------------
# 5. NORMALISE HYDROGRAPHS
# -------------------------------
norm_hydro = []

for h in hydrographs:
    if np.max(h) > 0:
        norm_hydro.append(h / np.max(h))

# ensure same length (truncate)
min_len = min(len(h) for h in norm_hydro)
norm_matrix = np.array([h[:min_len] for h in norm_hydro])

# -------------------------------
# 6. CREATE MEDIAN HYDROGRAPH (TARGET SHAPE)
# -------------------------------
median_shape = np.median(norm_matrix, axis=0)

# -------------------------------
# 7. FIND BEST MATCHING REAL EVENT
# -------------------------------
errors = []

for i in range(len(norm_matrix)):
    err = np.sum((norm_matrix[i] - median_shape)**2)
    errors.append(err)

best_index = np.argmin(errors)
best_event_id = event_ids[best_index]

# -------------------------------
# 8. EXTRACT BEST EVENT (ORIGINAL DATA)
# -------------------------------
best_event = df[df['event_id'] == best_event_id]

# -------------------------------
# 9. SAVE FOR HEC-RAS
# -------------------------------
output_file = r"C:\Hec_Ras\ProJect6_big\best_flood_event_for_HECRAS.csv"
best_event[['flvalue(m3/sec)']].to_csv(output_file)

# -------------------------------
# 10. PRINT SUMMARY
# -------------------------------
print("Best flood event selected!")
print("Event ID:", best_event_id)
print("Duration (days):", len(best_event))
print("Peak flow:", best_event['flvalue(m3/sec)'].max())
print("Saved to:", output_file)